# Evaluation and Benchmarking of Model Forecasts

This Source Code Form is subject to the terms of the Mozilla Public  License, v. 2.0. If a copy of the MPL was not distributed with this file, You can obtain one at http://mozilla.org/MPL/2.0/.
This notebook demonstrates how to evaluate and visualize the results of a previously trained **S4Casting** model.  
It covers:
1. Loading the saved inference output from disk.  
2. Reconstructing the forecast distribution.  
3. Computing key probabilistic and quantile-based metrics.  
4. Visualizing forecast behavior against ground truth data.

## Prerequisites

Before running this notebook:
- ✅ Complete `01_train_model.ipynb` (generates checkpoints)
- ✅ Complete `02_inference.ipynb` (generates `out/results` pickle file)

## Load Inference Results and Configuration

We start by loading the serialized inference results (`df_result.pkl`) produced during model inference.  
This file includes:
- The raw predictions: `predictions`
- The context input data: `X`
- The context mask data: `Xm`  
- Model configuration: `config`

From the configuration, we extract parameters such as:
- The sampling rate: `base_sample_interval_minutes`  
- The historical context window: `input_width_days`  
- The forecast horizon: `predict_width_days`

In [ ]:
import pathlib
import pickle
import warnings

from s4casting.core.config import Configuration

warnings.filterwarnings("ignore")

path = pathlib.Path("out/results")
with path.open("rb") as f:
    df_result = pickle.load(f)


config: Configuration = df_result["config"]

config.metrics.crps = True
config.metrics.mae = True
config.metrics.precision = True
config.metrics.recall = True
config.metrics.fbeta = True

predict_width_days = config.model.predict_width_days  # 2
input_sample_interval_minutes = config.model.input_sample_intervals_minutes[0]  # 15
context_steps_cfg = int(config.model.input_width_days * 24 * 60 / config.model.base_sample_interval_minutes)
predict_steps_cfg = int(config.model.predict_width_days * 24 * 60 / config.model.base_sample_interval_minutes)
output_sample_interval_minutes = config.model.output_sample_intervals_minutes[0]  # 15

print(
    "Context steps:",
    context_steps_cfg,
    "Predict steps:",
    predict_steps_cfg,
    "Total steps:",
    context_steps_cfg + predict_steps_cfg,
    "Patch size:",
    config.model.patch_encoder.patch_size,
    "Input interval (min):",
    input_sample_interval_minutes,
    "Output interval (min):",
    output_sample_interval_minutes,
)

## Construct the ground truth DataFrames
With the same sinusoidal data generation logic used during inference, we reconstruct the ground truth DataFrames for evaluation.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tests.utils import create_sinusoid_dataframe

# put timestamp as index
df = create_sinusoid_dataframe(n=864 + predict_steps_cfg, interval_min=15, start_time="2023-01-01 00:00:00").set_index(
    "timestamp"
)

print("DataFrame shape:", df.shape)

## Data Structure Overview

Before evaluating, it’s important to understand the four key tensors that define the forecasting setup:

| Variable | Shape | Description |
|-----------|--------|-------------|
| **X (Context / History)** | `(B, T_ctx, F_in)` | Past input sequence (historical data) used to condition the forecast. |
| **Xm (Context Mask)** | Same as `X` | Binary mask (1 = valid, 0 = missing) for context features. |
| **Y (Forecast Target)** | `(B, T_fore, F_tar)` | The ground-truth future values the model is trying to predict. |
| **Ym (Target Mask)** | Same as `Y` | Binary mask (1 = valid, 0 = missing) for forecast targets. |

Definitions of symbolic dimensions:
- `B`: Batch size (number of independent sequences processed together).
- `T_ctx`: Number of timesteps in the historical context window.
- `T_fore`: Number of timesteps in the forecast horizon.
- `F_in`: Number of input features per context timestep (e.g. load, weather variables).
- `F_tar`: Number of target features per forecast timestep (often 1 if predicting a single series).

We reconstruct tensors for evaluation:
- Extract predictions for the forecast horizon.
- Build ground truth (`Y`) and its validity mask (`Ym`).
- Identify the forecast start time from the first missing value in `load`.

In [ ]:
import pandas as pd
import torch

predictions = df_result["predictions"]
X = df_result["X"]
Xm = df_result["Xm"]

# Reconstruct ground truth for forecast horizon
Y_series = df[-predict_steps_cfg:]
Ym_series = (~Y_series.isna()).astype(int)

Y = torch.tensor(Y_series.to_numpy(), dtype=torch.float32).view(1, -1, 1)
Ym = torch.tensor(Ym_series.to_numpy(), dtype=torch.int32).view(1, -1, 1)

print(f"Predictions shape: {predictions.shape}")
print(f"Y shape: {Y.shape}")
print(f"Ym shape: {Ym.shape}")
print(f"X shape: {X.shape}")

# Forecast Visualization and Distribution Analysis

This plot visualizes the same probabilistic structure, 
which represents real-world evaluation under specific test conditions.
The model outputs **multiple quantile forecasts** (e.g., 5th, 25th, 50th, 75th, 95th percentiles), which together represent the full **predictive distribution**. 

- **Lower quantiles** (transparent green): Conservative estimates (5th-25th percentile).
- **Median quantile** (mid-opacity): Most likely forecast (50th percentile).
- **Upper quantiles** (opaque green): Optimistic estimates (75th-95th percentile).

### **How to Interpret**

**Good Calibration:**  
- Ground truth (black line) falls **within the quantile range** (green band).

**Poor Calibration:**  
- Ground truth consistently **above** or **below** all quantiles → model underestimates uncertainty.
- Quantiles are **too wide** → underconfident.

> Note: The set of quantiles used for forecasting is configured in your TOML under `config.model.output_head.quantile_values`. 

In [ ]:
from s4casting.visualisation.quantile_plotter import plot_short_term_training_quantiles

plot_short_term_training_quantiles(
    X=X, Xm=Xm, Y=Y, Ym=Ym, quantiles=predictions, quantile_values=config.model.output_head.quantile_values
)

## Forecast Evaluation Metrics

This cell evaluates the **probabilistic forecast performance** of the trained model using the `Metrics` class from `s4eval.metrics`.  
It summarizes both overall probabilistic accuracy and quantile-specific classification behavior.

### **1. Probabilistic Metrics**
These metrics assess the **calibration and sharpness** of the entire predictive distribution.

| Metric | What It Measures | Good Performance |
|---------|------------------|------------------|
| **CRPS** (Continuous Ranked Probability Score) | Distance between predicted cumulative distribution and actual outcome. Combines calibration (are probabilities accurate?) and sharpness (are predictions precise?). | **Lower is better**<br>Values closer to 0 indicate well-calibrated, sharp forecasts. |
| **NLL** (Negative Log-Likelihood) | How much probability mass the model assigns to the actual observed value. | **More negative is better**<br>Values closer to 0 or negative indicate confident, accurate probability assignments. |

<br>

> **Note:** The NLL (loss) metric is currently not computed in this evaluation because we have not yet implemented the loss calculation directly from quantile predictions without the model. This metric is tracked during training but requires the full model forward pass to compute. For post-inference evaluation, we rely on CRPS and quantile-specific metrics instead.

### **2. Quantile-Based Metrics**
Each predicted quantile (e.g., 0.05, 0.25, 0.50, 0.75, 0.95) is evaluated as a **binary classifier** based on a congestion threshold (`climits`).

| Column | Definition | Interpretation |
|---------|------------|----------------|
| **Precision** | Of all predicted exceedances, what fraction were correct? | High precision = few false alarms |
| **Recall** | Of all actual exceedances, what fraction did we detect? | High recall = few missed events |
| **Fβ** | Weighted harmonic mean of precision and recall (β=10 emphasizes recall) | Balances detection vs false alarms, prioritizing event capture |
| **MAE** | Mean absolute error between predicted quantile values and observations | Lower MAE = more accurate point estimates |

In [ ]:
from s4casting.eval.metrics import Metrics

metrics = Metrics(
    output_sample_interval_minutes=output_sample_interval_minutes,
    input_sample_interval_minutes=input_sample_interval_minutes,
    climits=4,
    quantiles=predictions,
    Y=Y,
    model_config=config.model,
    metrics_config=config.metrics,
    sign=None,
    quantile_values=config.model.output_head.quantile_values,
    loss=0.0,
)

metrics_df = pd.DataFrame(metrics.get_metrics())
metrics_df

## Evaluation Results
The model shows a CRPS of 0.15456. The loss column (NLL) remains zero because NLL is not computed in this evaluation.

For the quantile-based metrics, precision is basically perfect (often 1.0), and recall gets closer to 1.0 as you move toward the middle quantiles, so Fβ scores are very strong and the model hardly misses any events. The MAE is lowest around the 0.5 quantile, so the median forecast is the most accurate, with slightly bigger errors at the extreme quantiles (0.01 and 0.99). Overall, the model gives sharp, reliable forecasts and behaves especially well around the central quantiles.

## Summary & Next Steps

You've successfully evaluated and visualized the performance of a trained S4Casting model!

### Next Steps
1. **Tune hyperparameters** — Adjust different parameters in the configuration file to improve performance
2. **Test on real data** — Replace synthetic sinusoid with actual load profiles
3. **Benchmark alternatives** — Test against other benchmarks like STEFbeam or our own benchmarks.
